# Orca Nano — QLoRA Fine-Tune v2 (Colab free T4)

Run cells top to bottom. Before starting: **Runtime → Change runtime type → T4 GPU**.

**What changed from v1:**
- v1 scored *worse* than the untrained base model (0.718 vs 0.76 baseline) —
  the validation loss rose after step 100 while training loss kept dropping,
  a real overfitting signal on a small (653-example) dataset trained for 3
  epochs.
- Dataset is now much larger (2050 train / 108 eval — the full nano
  distillation batch, not a partial slice) and includes a new targeted
  `honesty_hedging` domain (150 examples) that explicitly teaches
  admitting uncertainty on genuinely unknowable questions — v1's biggest
  score drop was exactly this cluster, likely because the shared
  anti-sycophancy prompt rules used elsewhere discourage hedging entirely.
- **Epochs reduced 3 → 2** to reduce overfitting risk on this dataset size.
- Export now copies straight to Google Drive instead of using
  `files.download()` — that method failed 3 times in a row last round from
  page reloads killing the in-browser transfer mid-download.

In [ ]:
!pip install -q "unsloth[colab-new] @ git+https://github.com/unslothai/unsloth.git" trl transformers datasets peft bitsandbytes accelerate

## Upload your training data

Upload the two files from your Desktop: `orca_llama3_train.jsonl` and `orca_llama3_eval.jsonl`
(already copied there — 2050 train / 108 eval examples).

In [ ]:
from google.colab import files
uploaded = files.upload()  # select orca_llama3_train.jsonl and orca_llama3_eval.jsonl from Desktop
print('Uploaded:', list(uploaded.keys()))

In [ ]:
import json

def load_jsonl(path):
    lines = []
    with open(path) as f:
        for line in f:
            line = line.strip()
            if line:
                try:
                    lines.append(json.loads(line))
                except Exception:
                    pass
    return lines

raw_train = load_jsonl('orca_llama3_train.jsonl')
raw_eval  = load_jsonl('orca_llama3_eval.jsonl') if 'orca_llama3_eval.jsonl' in uploaded else raw_train[:max(1, len(raw_train)//10)]
print(f'train={len(raw_train)} eval={len(raw_eval)}')

## Load base model (4-bit) + attach LoRA

Rank 16 (not 128 like the A100 preset) — T4 has 16GB VRAM, a bigger rank
risks an out-of-memory crash partway through training.

In [ ]:
from unsloth import FastLanguageModel
import torch

max_seq_length = 2048  # smaller than the 4096 cloud preset — fits T4 memory
base_model = "unsloth/Qwen2.5-7B-Instruct"

model, tokenizer = FastLanguageModel.from_pretrained(
    model_name=base_model,
    max_seq_length=max_seq_length,
    dtype=None,
    load_in_4bit=True,
)

model = FastLanguageModel.get_peft_model(
    model,
    r=16,
    lora_alpha=32,
    lora_dropout=0.05,
    target_modules=["q_proj", "k_proj", "v_proj", "o_proj", "gate_proj", "up_proj", "down_proj"],
    bias="none",
    use_gradient_checkpointing="unsloth",
    random_state=42,
)

In [ ]:
from datasets import Dataset

def format_conv(ex):
    turns = ex.get("conversations", ex.get("text"))
    if isinstance(turns, str):
        return turns  # already-formatted llama3 text field
    parts = []
    for t in turns:
        role = t.get("role", "")
        val  = t.get("value", "")
        if role == "system":
            parts.append(f"<|start_header_id|>system<|end_header_id|>\n\n{val}<|eot_id|>")
        elif role == "human":
            parts.append(f"<|start_header_id|>user<|end_header_id|>\n\n{val}<|eot_id|>")
        elif role == "gpt":
            parts.append(f"<|start_header_id|>assistant<|end_header_id|>\n\n{val}<|eot_id|>")
    return "".join(parts)

# The formatter.py output already has a 'text' field per example (llama3 format) — use it directly if present.
train_ds = Dataset.from_list([{"text": ex["text"] if "text" in ex else format_conv(ex)} for ex in raw_train])
eval_ds  = Dataset.from_list([{"text": ex["text"] if "text" in ex else format_conv(ex)} for ex in raw_eval])
print(f'train_ds={len(train_ds)} eval_ds={len(eval_ds)}')

## Train

Batch size 2 + grad accumulation 4 (effective batch 8) — smaller than the
cloud preset's batch 8, again sized for T4's 16GB rather than an A100's 40GB.

**2 epochs, not 3** — v1's validation loss rose after step 100 while
training loss kept dropping, a real overfitting signal this dataset size
doesn't yet support 3 full passes over.

In [ ]:
from trl import SFTTrainer
from transformers import TrainingArguments
import time

trainer = SFTTrainer(
    model=model,
    tokenizer=tokenizer,
    train_dataset=train_ds,
    eval_dataset=eval_ds,
    dataset_text_field="text",
    max_seq_length=max_seq_length,
    args=TrainingArguments(
        per_device_train_batch_size=2,
        gradient_accumulation_steps=4,
        num_train_epochs=2,
        learning_rate=2e-4,
        warmup_ratio=0.05,
        lr_scheduler_type="cosine",
        weight_decay=0.01,
        max_grad_norm=1.0,
        fp16=not torch.cuda.is_bf16_supported(),
        bf16=torch.cuda.is_bf16_supported(),
        logging_steps=10,
        eval_steps=100,
        save_strategy="no",
        output_dir="output",
        eval_strategy="steps",
        report_to="none",
    ),
)

print("[train] starting QLoRA training...")
t0 = time.time()
trainer.train()
elapsed = (time.time() - t0) / 60
print(f"[train] done in {elapsed:.1f} min")

## Merge LoRA + export GGUF

In [ ]:
print("[merge] merging LoRA adapters...")
model.save_pretrained_merged("merged", tokenizer, save_method="merged_16bit")
print("[merge] saved to ./merged")

print("[gguf] converting to GGUF q4_k_m...")
model.save_pretrained_gguf("gguf", tokenizer, quantization_method="q4_k_m")
print("[gguf] saved to ./gguf")

## Copy to Google Drive, then download from drive.google.com

Skipping `files.download()` entirely this time — it requires a continuously
live connection through the browser tab and failed 3 times in a row last
round from page reloads. Copying to Drive is a normal, resumable transfer;
download the file from drive.google.com afterward like any other file.

In [ ]:
import glob
import shutil
from google.colab import drive

drive.mount('/content/drive')

# Recursive + case-insensitive search — unsloth's output folder naming has
# varied between runs (e.g. 'gguf/' vs 'gguf_gguf/'), so search broadly
# instead of assuming one exact path.
candidates = [f for f in glob.glob('**/*.gguf', recursive=True) if 'q4_k_m' in f.lower()]
print('Found:', candidates)

if candidates:
    source_path = candidates[0]
    filename = source_path.split('/')[-1]
    dest_path = f'/content/drive/MyDrive/{filename}'
    print(f'Copying {filename} to Google Drive...')
    shutil.copy(source_path, dest_path)
    print(f'Done! File saved to: {dest_path}')
    print('Now go to drive.google.com and download it from the root of My Drive.')
else:
    print('No GGUF file found — check the [gguf] step above for errors.')